# UniChart — Custom `color_map`

Demonstrates and verifies the user-customizable **`color_map`** — the ordered
list of colors assigned to datasets by index (the color analogue of
`marker_map`).

Covered:

- **Default** map = the Plotly qualitative palette.
- **Custom map set before load** → datasets cycle through your colors.
- **Expectation management**: setting `color_map` on an *existing* notebook does
  **not** retroactively recolor already-loaded datasets — colors are assigned at
  load. `reset_format()` (or loading more data) applies the new map.
- **Integer `color()` lookups** resolve through the custom map.
- **Empty-map guard** falls back to the Plotly palette (no crash).
- Using a **built-in Plotly qualitative list** as a map.

Each section makes hard `assert`s (the real test) and renders a plot for visual
confirmation. The summary prints **ALL COLOR_MAP TESTS PASSED** if every
assertion held.

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
import plotly.express as px
from unichart import UnichartNotebook

PLOTLY = list(px.colors.qualitative.Plotly)
# A short 3-color custom map, so cycling is visible across 6 datasets.
CUSTOM = ['#E41A1C', '#377EB8', '#4DAF4A']   # red / blue / green

PASSED = 0
def check(cond, msg):
    global PASSED
    assert cond, 'FAILED: ' + msg
    PASSED += 1
    print('PASS:', msg)

def make_nb(n=6, color_map=None):
    """n datasets of y = x + offset; optionally set color_map BEFORE loading."""
    nb = UnichartNotebook()
    if color_map is not None:
        nb.color_map = list(color_map)
    x = np.linspace(0, 10, 25)
    for i in range(n):
        nb.load_df(pd.DataFrame({'x': x, 'y': x + i * 2}), title=f'S{i}')
    nb.linestyle('all', '-')   # draw lines so color bands are easy to see
    return nb

def expected(cmap, n):
    return [cmap[i % len(cmap)] for i in range(n)]

print('default color_map (first 4):', UnichartNotebook().color_map[:4])
print('CUSTOM map:', CUSTOM)

UniChart Notebook Environment Initialized.
default color_map (first 4): ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']
CUSTOM map: ['#E41A1C', '#377EB8', '#4DAF4A']


## A. Default map = Plotly palette

Out of the box, datasets are colored from `px.colors.qualitative.Plotly`.

In [2]:
uc = make_nb(6)
colors = [d.color for d in uc.sets]
print('dataset colors:', colors)
check(colors == expected(PLOTLY, 6), "default datasets use the Plotly palette by index")
uc.plot('x', 'y', suptitle='A. Default color_map (Plotly palette)').show()

UniChart Notebook Environment Initialized.
Loaded Set 0: S0
Loaded Set 1: S1
Loaded Set 2: S2
Loaded Set 3: S3
Loaded Set 4: S4
Loaded Set 5: S5
dataset colors: ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3']
PASS: default datasets use the Plotly palette by index


## B. Custom map set before loading

Setting `color_map` to your own list before loading makes datasets cycle through
it. With a 3-color map and 6 datasets, the colors repeat: red, blue, green, red,
blue, green.

In [ ]:
uc.color_map = ['#E41A1C', '#377EB8',]
uc.plot('x', 'y', suptitle='B. Custom color_map (red / blue / green, cycling)').show()

NameError: name 'x' is not defined

In [ ]:
uc = make_nb(6, color_map=CUSTOM)
colors = [d.color for d in uc.sets]
print('dataset colors:', colors)
check(colors == expected(CUSTOM, 6), "custom map set before load cycles across datasets")
check(colors == ['#E41A1C', '#377EB8', '#4DAF4A', '#E41A1C', '#377EB8', '#4DAF4A'],
      "6 datasets over a 3-color map repeat red/blue/green")
uc.plot('x', 'y', suptitle='B. Custom color_map (red / blue / green, cycling)').show()

UniChart Notebook Environment Initialized.
Loaded Set 0: S0
Loaded Set 1: S1
Loaded Set 2: S2
Loaded Set 3: S3
Loaded Set 4: S4
Loaded Set 5: S5
dataset colors: ['#E41A1C', '#377EB8', '#4DAF4A', '#E41A1C', '#377EB8', '#4DAF4A']
PASS: custom map set before load cycles across datasets


AssertionError: FAILED: 6 datasets over a 3-color map repeat red/blue/green

## C. Expectation management — applying a new map to an existing notebook

`color_map` is read when a dataset is created. Reassigning it later does **not**
retroactively recolor existing datasets — you must `reset_format()` (or load new
data) for the new map to take effect. This cell proves both halves.

In [4]:
uc = make_nb(6)                       # loaded with the default Plotly map
uc.plot('x', 'y', suptitle='C1. Before: default colors').show()

uc.color_map = CUSTOM                  # change the map AFTER loading
check([d.color for d in uc.sets] == expected(PLOTLY, 6),
      "reassigning color_map does NOT retroactively recolor existing datasets")

uc.reset_format()                     # now apply it
check([d.color for d in uc.sets] == expected(CUSTOM, 6),
      "reset_format() re-applies the (custom) color_map to existing datasets")
uc.linestyle('all', '-')
uc.plot('x', 'y', suptitle='C2. After reset_format(): custom colors').show()

UniChart Notebook Environment Initialized.
Loaded Set 0: S0
Loaded Set 1: S1
Loaded Set 2: S2
Loaded Set 3: S3
Loaded Set 4: S4
Loaded Set 5: S5


PASS: reassigning color_map does NOT retroactively recolor existing datasets
Reset: dataset formatting, variable formats, lines, highlights, axis limits, font sizes.
PASS: reset_format() re-applies the (custom) color_map to existing datasets


## D. Integer `color()` lookups use the custom map

`color(target, <int>)` resolves the integer through `color_map`, so a dataset can
borrow another index's mapped color.

In [16]:
print(uc.color_map[10])

IndexError: list index out of range

In [17]:
uc = make_nb(6, color_map=["red", "red", "blue", "blue", "green", "green"])
uc.plot('x', 'y', suptitle='D. Custom color_map with duplicates').show()

UniChart Notebook Environment Initialized.
Loaded Set 0: S0
Loaded Set 1: S1
Loaded Set 2: S2
Loaded Set 3: S3
Loaded Set 4: S4
Loaded Set 5: S5


In [25]:
uc.box('x','y')

In [5]:
uc = make_nb(6, color_map=CUSTOM)
uc.color(0, 4)   # index 4 -> CUSTOM[4 % 3] = CUSTOM[1] = blue
print('set 0 color after color(0, 4):', uc.sets[0].color)
check(uc.sets[0].color == CUSTOM[4 % len(CUSTOM)], "int color() resolves through the custom map")
check(uc.sets[0].color == '#377EB8', "color(0, 4) -> blue (CUSTOM[1])")

UniChart Notebook Environment Initialized.
Loaded Set 0: S0
Loaded Set 1: S1
Loaded Set 2: S2
Loaded Set 3: S3
Loaded Set 4: S4
Loaded Set 5: S5
set 0 color after color(0, 4): #377EB8
PASS: int color() resolves through the custom map
PASS: color(0, 4) -> blue (CUSTOM[1])


## E. Empty-map guard

If `color_map` is set to an empty list, lookups safely fall back to the Plotly
palette instead of raising.

In [6]:
uc = make_nb(3)
uc.color_map = []
check(uc._color_at(2) == PLOTLY[2], "_color_at falls back to Plotly when color_map is empty")
# and loading a new dataset with an empty map does not crash
uc.load_df(pd.DataFrame({'x': [1], 'y': [1]}), title='extra')
check(uc.sets[-1].color == PLOTLY[3], "new dataset under empty map falls back to Plotly[index]")
print('no crash; fallback color:', uc.sets[-1].color)

UniChart Notebook Environment Initialized.
Loaded Set 0: S0
Loaded Set 1: S1
Loaded Set 2: S2
PASS: _color_at falls back to Plotly when color_map is empty
Loaded Set 3: extra
PASS: new dataset under empty map falls back to Plotly[index]
no crash; fallback color: #AB63FA


## F. Using a built-in Plotly qualitative list as a map

Any list of color specs works — including Plotly's own named qualitative
sequences.

In [7]:
SET2 = list(px.colors.qualitative.Set2)
uc = make_nb(6, color_map=SET2)
check([d.color for d in uc.sets] == expected(SET2, 6), "a Plotly qualitative list works as a color_map")
uc.plot('x', 'y', suptitle='F. color_map = px.colors.qualitative.Set2').show()

UniChart Notebook Environment Initialized.
Loaded Set 0: S0
Loaded Set 1: S1
Loaded Set 2: S2
Loaded Set 3: S3
Loaded Set 4: S4
Loaded Set 5: S5
PASS: a Plotly qualitative list works as a color_map


## Summary

In [8]:
print(f'{PASSED} assertions passed.')
print('ALL COLOR_MAP TESTS PASSED' if PASSED >= 9 else 'CHECK COUNT UNEXPECTED')

10 assertions passed.
ALL COLOR_MAP TESTS PASSED
